In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==5.0.0
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel

MODEL_ID = "unsloth/Ministral-3-3B-Instruct-2512"
# MODEL_ID = "unsloth/Ministral-3-8B-Instruct-2512"
# MODEL_ID = "unsloth/Ministral-3-14B-Instruct-2512"
MAX_SEQ_LENGTH = 8192 # Unsloth handles long context very efficiently

print("Loading Unsloth model...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True, # 4-bit quantization,
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for long context
)

# Since your JSON rows contain "<s>" and "</s>" explicitly:
tokenizer.add_bos_token = False
tokenizer.add_eos_token = False
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

In [ ]:
# --- 3. Add LoRA Adapters ---
print("Applying LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 32,
    lora_dropout = 0, # Unsloth recommends 0 dropout for optimized kernels
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Uses Unsloth's optimized checkpointing
    random_state = 3407,
)



In [ ]:
from datasets import load_dataset
from src.prompts import get_train_prompt_v5


DATA_FILE = "../data/processed/train-v6.jsonl" 

# --- 4. Load Dataset ---
print("Loading dataset...")
dataset = load_dataset("json", data_files=DATA_FILE, split="train")
dataset = dataset.train_test_split(test_size=0.1, seed=3407, shuffle=True) # Use 90% for training, 10% for validation

dataset = dataset.map(
    lambda x: {
        "text": [get_train_prompt_v5(c, o) for c, o in zip(x["corrupted"], x["original"])]
    }, 
    batched=True
)

In [ ]:
dataset

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bf16_supported


# --- 5. Training Arguments ---
print("Configuring trainer...")
sft_config = SFTConfig(
    output_dir = "./results",
    dataset_text_field = "text",
    max_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False, # Can set to True for speed boost if data allows

    # --- Training Parameters ---
    per_device_train_batch_size = 4,
    gradient_accumulation_steps = 4,
    num_train_epochs = 1,
    learning_rate = 1e-4, # 2e-4

    fp16 = not is_bf16_supported(),
    bf16 = is_bf16_supported(), # True for A6000

    logging_steps = 25,
    save_strategy = "steps",
    save_steps = 100,

    warmup_steps = 5,
    optim = "adamw_8bit", # 8-bit optimizer saves even more memory
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    seed = 3407,
    report_to = "none",
    torch_compile=False,
    torch_compile_backend=None,
    torch_compile_mode=None,

    eval_accumulation_steps = 4,
    eval_strategy = "steps",
    eval_steps = 25,
)

# --- 6. Initialize Trainer ---
trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    args = sft_config,
    train_dataset = dataset["train"],
    eval_dataset = dataset["test"],
)

In [ ]:
# --- 7. Start Training ---
print("Starting training...")
trainer.train()

In [ ]:
NEW_MODEL_NAME = f"../models/{MODEL_ID.split("/")[1]}-GEC-v6"

print(f"Saving model to {NEW_MODEL_NAME}...")
model.save_pretrained(NEW_MODEL_NAME) # Saves adapters
tokenizer.save_pretrained(NEW_MODEL_NAME)

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "./models/Ministral-3-14B-Instruct-2512-GEC-v6", # Point directly to the ADAPTER folder
    max_seq_length = 8192,          # Use the same as your training
    load_in_4bit = True,            # Set to True if you trained with 4-bit
    dtype = None,                   # Auto-detect (Float16/Bfloat16)
    device_map="cpu"
)

In [ ]:
from src.prompts import get_inference_prompt_v5

# Switch model to inference mode (merges LoRA weights for generation)
FastLanguageModel.for_inference(model)

test = get_inference_prompt_v5("Ich habe dich gesagt, dass du eher ein Tasse Tee kaufen sollen.")
test

In [ ]:
ten = tokenizer(None, test, return_tensors="pt", add_special_tokens=False).to(model.device)
ten

In [ ]:
output = model.generate(**ten, max_new_tokens=128)

In [ ]:
output

In [ ]:
# Decode only the NEW tokens (skip the input prompt tokens)
input_length = ten["input_ids"].shape[1]
generated_tokens = output[0][input_length:]
output_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
print("Model Output:")
print(output_text)

In [ ]:
# To save as a 16-bit merged model
MERGE_SAVE_PATH = f"../models/merged_16bit/{MODEL_ID.split("/")[1]}-GEC-v6"

model.save_pretrained_merged(MERGE_SAVE_PATH, tokenizer, save_method = "merged_16bit")